# Modelos de Lenguaje, formato de chat y tokens

En este notebook se muestra como integrar la API de OpenAI mediante una API key. Además se implementa una función auziliar para enviar un prompt al modelo gpt-3.5-turbo y recibir una respuesta directa. Se explica también la utilidad de los parámetros system y user a la hora de usar un modelo LLM.


## 1. Configuración
#### Carga de la API key y bibliotecas de python necesarias.


In [1]:
import os
import openai
import tiktoken
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) #lectura en local del archivo .env para cargar la api key de forma segura

openai.api_key  = os.environ['OPENAI_API_KEY']

#### Función auxiliar


In [2]:
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0, #grado de aleatoriedad de la salida del modelo
    )
    return response.choices[0].message["content"]

#A continuacion se detalla una version de la funcion auxiliar para poder usar la version 1.0.0 de la libreria de OpenAI
client = openai.OpenAI()

def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0
    )
    return response.choices[0].message.content
```

## 2. Enviar un prompt al modelo y obtener una respuesta

In [3]:
response = get_completion("Cuál es la capital de Andalucía?")

In [4]:
print(response)

La capital de Andalucía es Sevilla.


## 3. Tokens

In [5]:
response = get_completion("Devuelve el texto lollipop \
y en orden inverso")
print(response)

poppilol


"lollipop" in reverse should be "popillol"

In [6]:
response = get_completion("""Devuelve el texto \
l-o-l-l-i-p-o-p y en orden inverso""")

In [7]:
response

'p-o-p-i-l-l-o-l'

## 4. Función auxiliar (formato de chat)


In [8]:
def get_completion_from_messages(messages, 
                                 model="gpt-3.5-turbo", 
                                 temperature=0, 
                                 max_tokens=500):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature, #este es el grado de aleatoriedad de la salida del modelo
        max_tokens=max_tokens, #cantidad máxima de tokens que el modelo puede emitir 
    )
    return response.choices[0].message["content"]

Tal y como mencionamos antes, a la hora de responder, un modelo LLM tiene en cuenta estos parámetros:
- system: el estilo  o reglas globales indicados por el mensaje del sistema.
- user: la petición concreta del usuario.

In [9]:
messages =  [  
{'role':'system', 
 'content':"""Eres un asistente que responde con el estilo de un argentino."""},    
{'role':'user', 
 'content':"""Escribe un poema muy corto/ sobre una zanahoria feliz."""},  
] 
response = get_completion_from_messages(messages, temperature=1)
print(response)

En la huerta radiante brilla su color,
la zanahoria feliz baila con mucho ardor. 


In [10]:
#longitud
messages = [  
    {
        'role': 'system',
        'content': 'Todas tus respuestas deben ser \
de una sola oración.'
    },    
    {
        'role': 'user',
        'content': 'Escríbeme una historia sobre una zanahoria feliz'
    },  
] 

response = get_completion_from_messages(messages, temperature=1)
print(response)

Había una zanahoria llamada Zanito que saltaba de alegría cada vez que veía el sol brillar.


In [11]:
#combinamos
messages =  [  
{'role':'system',
 'content':"""Eres un asistente que \
responde con el estilo de un argentino. \
Todas tus respuestas deben ser de una sola oración."""},    
{'role':'user',
 'content':"""Escribe un poema muy corto sobre una zanahoria feliz"""},
] 
response = get_completion_from_messages(messages, 
                                        temperature =1)
print(response)

En la huerta brillante, la zanahoria sonríe contenta.


In [12]:
def get_completion_and_token_count(messages, 
                                   model="gpt-3.5-turbo", 
                                   temperature=0, 
                                   max_tokens=500):
    
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature, 
        max_tokens=max_tokens,
    )
    
    content = response.choices[0].message["content"]
    
    token_dict = {
'prompt_tokens':response['usage']['prompt_tokens'],
'completion_tokens':response['usage']['completion_tokens'],
'total_tokens':response['usage']['total_tokens'],
    }

    return content, token_dict

In [13]:
messages = [
{'role':'system', 
 'content':"""Eres un asistente que
responde con el estilo de un argentino."""},    
{'role':'user',
 'content':"""Escribe un poema muy corto/ sobre una zanahoria feliz."""},  
] 
response, token_dict = get_completion_and_token_count(messages)

In [14]:
print(response)

En la huerta baila contenta,
la zanahoria sonriente brilla,
con su color naranja reluciente,
alegra la tierra con su chispa argentina.


In [15]:
print(token_dict)

{'prompt_tokens': 46, 'completion_tokens': 43, 'total_tokens': 89}
